# 🤖 Delay Prediction Model — XGBoost
**Project:** Supply Chain Optimizer | 3PL Logistics Intelligence

**Objective:** Train a binary classifier to predict whether a shipment will be delayed.

**Model:** XGBoost with engineered features, class balancing, and cross-validation.

**Target:** `is_delayed` (0 = On-Time, 1 = Delayed)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score
)
import xgboost as xgb
import joblib
import json
import os

plt.rcParams.update({
    'figure.facecolor':  '#060a0f',
    'axes.facecolor':    '#0a0e14',
    'axes.edgecolor':    '#1a2535',
    'axes.labelcolor':   '#e2e8f0',
    'text.color':        '#e2e8f0',
    'xtick.color':       '#4a5568',
    'ytick.color':       '#4a5568',
    'grid.color':        '#0f1820',
    'grid.linestyle':    '--',
    'font.family':       'monospace',
})

ACCENT = '#e8770a'
GREEN  = '#4ade80'
BLUE   = '#60a5fa'
RED    = '#f87171'
PURPLE = '#a78bfa'

print('✅ Libraries loaded')

## 1. Load & Feature Engineer

In [ ]:
df = pd.read_csv('../data/raw/shipments_3yr.csv')

# Encode categorical
le = LabelEncoder()
df['cargo_type_enc'] = le.fit_transform(df['cargo_type'])

# Cyclical time encoding
df['hour_sin']  = np.sin(2 * np.pi * df['departure_hour'] / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['departure_hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

FEATURES = [
    'weather_score', 'traffic_index', 'route_complexity', 'distance_km',
    'driver_experience', 'driver_rating', 'departure_hour', 'day_of_week',
    'month', 'is_holiday', 'is_weekend', 'weight_kg', 'cargo_type_enc',
    'planned_hours', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
]
TARGET = 'is_delayed'

X = df[FEATURES]
y = df[TARGET]

print('=' * 55)
print('  DATASET SUMMARY')
print('=' * 55)
print(f'  Total records:  {len(X):,}')
print(f'  Features:       {len(FEATURES)}')
print(f'  Delay rate:     {y.mean()*100:.1f}%  ({y.sum():,} delayed)')
print(f'  On-time:        {(1-y.mean())*100:.1f}%  ({(y==0).sum():,} on-time)')
print(f'  Class ratio:    1 : {(y==0).sum()/y.sum():.1f}')
print('=' * 55)

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set:  {len(X_train):,} records ({y_train.mean()*100:.1f}% delay rate)')
print(f'Test set:   {len(X_test):,}  records ({y_test.mean()*100:.1f}% delay rate)')

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos:.2f}')

## 3. Train XGBoost Model

In [ ]:
model = xgb.XGBClassifier(
    n_estimators        = 400,
    max_depth           = 6,
    learning_rate       = 0.05,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    scale_pos_weight    = scale_pos,
    use_label_encoder   = False,
    eval_metric         = 'auc',
    random_state        = 42,
    n_jobs              = -1,
)

print('⟳ Training XGBoost...')
model.fit(
    X_train, y_train,
    eval_set    = [(X_train, y_train), (X_test, y_test)],
    verbose     = 50,
)
print('✅ Training complete')

## 4. Model Evaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc  = average_precision_score(y_test, y_proba)

print('=' * 55)
print('  MODEL PERFORMANCE')
print('=' * 55)
print(f'  ROC-AUC:  {roc_auc:.4f}')
print(f'  PR-AUC:   {pr_auc:.4f}')
print('=' * 55)
print()
print(classification_report(y_test, y_pred, target_names=['On-Time', 'Delayed']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('MODEL EVALUATION — XGBOOST DELAY PREDICTOR', fontsize=14, color='#e2e8f0')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color=ACCENT, lw=2.5, label=f'ROC AUC = {roc_auc_val:.3f}')
axes[0].plot([0,1],[0,1], '--', color='#4a5568', linewidth=1)
axes[0].fill_between(fpr, tpr, alpha=0.1, color=ACCENT)
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')
axes[0].grid(True)

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec, prec, color=BLUE, lw=2.5, label=f'PR AUC = {pr_auc:.3f}')
axes[1].fill_between(rec, prec, alpha=0.1, color=BLUE)
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True)

# Confusion Matrix
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['On-Time', 'Delayed'])
disp.plot(ax=axes[2], cmap='Blues', colorbar=False)
axes[2].set_title('Confusion Matrix')
axes[2].tick_params(colors='#e2e8f0')

plt.tight_layout()
plt.savefig('../docs/model_evaluation.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()

## 5. Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature':    FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('TOP 10 FEATURE IMPORTANCES:')
for _, row in fi.head(10).iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {row['feature']:<25} {bar:<20} {row['importance']:.4f}")

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#060a0f')

top10  = fi.head(10)
colors = [ACCENT if i == 0 else BLUE if i < 3 else '#4a5568'
          for i in range(len(top10))]

bars = ax.barh(top10['feature'][::-1], top10['importance'][::-1],
               color=colors[::-1], edgecolor='none', height=0.6)

for bar, val in zip(bars, top10['importance'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, color='#e2e8f0')

ax.set_title('XGBOOST FEATURE IMPORTANCE (Top 10)', color='#e2e8f0', pad=15, fontsize=13)
ax.set_xlabel('Importance Score')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()

## 6. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print('5-FOLD CROSS-VALIDATION RESULTS:')
for i, score in enumerate(cv_scores):
    bar = '█' * int(score * 50)
    print(f'  Fold {i+1}: {bar} {score:.4f}')
print(f'\n  Mean:  {cv_scores.mean():.4f}')
print(f'  Std:   {cv_scores.std():.4f}')
print(f'  Range: {cv_scores.min():.4f} – {cv_scores.max():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#060a0f')
ax.bar([f'Fold {i+1}' for i in range(5)], cv_scores,
       color=[GREEN if s > cv_scores.mean() else BLUE for s in cv_scores],
       edgecolor='none')
ax.axhline(cv_scores.mean(), color=ACCENT, linestyle='--', linewidth=2,
           label=f'Mean = {cv_scores.mean():.4f}')
ax.set_title('5-FOLD CROSS-VALIDATION ROC-AUC', color='#e2e8f0')
ax.set_ylabel('ROC-AUC Score')
ax.set_ylim(0.4, 1.0)
ax.legend()
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('../docs/cross_validation.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()

## 7. Save Model & Metrics

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(model, '../models/delay_predictor.pkl')
joblib.dump(le,    '../models/label_encoder.pkl')

fi.to_csv('../models/feature_importance.csv', index=False)

metrics = {
    'roc_auc':            round(float(roc_auc), 4),
    'pr_auc':             round(float(pr_auc),  4),
    'cv_roc_auc_mean':    round(float(cv_scores.mean()), 4),
    'cv_roc_auc_std':     round(float(cv_scores.std()),  4),
    'n_train':            int(len(X_train)),
    'n_test':             int(len(X_test)),
    'delay_rate':         round(float(y.mean()), 4),
    'features':           FEATURES,
    'model':              'XGBoostClassifier',
    'n_estimators':       400,
    'max_depth':          6,
    'learning_rate':      0.05,
}

with open('../models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('=' * 55)
print('  MODEL SAVED')
print('=' * 55)
print('  models/delay_predictor.pkl')
print('  models/label_encoder.pkl')
print('  models/feature_importance.csv')
print('  models/metrics.json')
print('\n  Final Metrics:')
print(f'    ROC-AUC:  {roc_auc:.4f}')
print(f'    PR-AUC:   {pr_auc:.4f}')
print(f'    CV Mean:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 55)

## 8. Sample Predictions

In [ ]:
sample = X_test.head(10).copy()
sample['actual']       = y_test.head(10).values
sample['predicted']    = model.predict(X_test.head(10))
sample['delay_prob']   = model.predict_proba(X_test.head(10))[:, 1].round(3)
sample['correct']      = (sample['actual'] == sample['predicted'])

display_cols = ['weather_score', 'traffic_index', 'route_complexity',
                'driver_experience', 'actual', 'predicted', 'delay_prob', 'correct']

print('SAMPLE PREDICTIONS (first 10 test records):')
print(sample[display_cols].to_string(index=False))
print(f'\nAccuracy on sample: {sample["correct"].mean()*100:.0f}%')